# Code to generate reference 3D ellipsoid STL files for printing/measurements

In [1]:
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pyvista as pv
import trimesh
from copy import deepcopy
import pymeshfix as mf
import cadquery as cq
from cq_warehouse.thread import PlasticBottleThread
%matplotlib widget

## Now generate different ellipsoids

In [2]:
#Use pyvista to generate an oblate spheroid. Increase u_res, v_res, w_res will increase file size.
#https://docs.pyvista.org/api/utilities/_autosummary/pyvista.parametricellipsoid
ellipsoid_surf = pv.ParametricEllipsoid(xradius = 30, yradius=30, zradius=12)

#triangulate
ellipsoid_surf_x2y2z1 = ellipsoid_surf.triangulate()



## Plot the results

In [3]:
# pl = pv.Plotter(shape=(1, 2))
# pl.subplot(0, 1)
# _ = pl.add_mesh(ellipsoid_surf ,color='blue')
# pl.show()

## Write stuff out

In [4]:
# Convert pyvista polydata to trimesh and export
ellipsoid_tm = trimesh.Trimesh(vertices=ellipsoid_surf.points, faces=ellipsoid_surf.faces.reshape(-1, 4)[:, 1:])

# Export to STL files
ellipsoid_tm.export('ellipsoid_surf.stl');


## Set up parameters for screw thread

In [5]:
# ======================= PARAMETERS =====================================
INPUT_STL          = "ellipsoid_surf.stl"
OUTPUT_TOP_STL     = "ellipsoid_surf_top.stl"
OUTPUT_BOTTOM_STL  = "ellipsoid_surf_bottom.stl"
 
CAVITY_CENTER      = np.array([0.0, 0.0, 0.0]) # coordinates are 0 in the center
CAVITY_RADIUS      = 9.0 #spherical hole for instrumentation, in mm
CUT_PLANE_NORMAL   = np.array([0.0, 0.0, 1.0]) # cutting in the vertical
CUT_PLANE_POINT    = CAVITY_CENTER
 
THREAD_SIZE        = "M24SP400"  #standard that corresponds to the diameter of the screw.
                                 #24 is the diameter here. 400 corresponds to the finish;
                                 #different finishes have different available diameters.
                                 # see thread.py in the cq_warehouse package.
THREAD_LENGTH      = 7.0 # depth (or height) of the screw. Also the size of the hole
                         # into which the screw goes. mm. 
WALL_THICKNESS     = 2.0 # width of the wall around the screw hole. mm
MANUFACTURING_COMP = 0.20 # mm print clearance - start at 0.2, tune from test prints
 
EXPORT_TOLERANCE         = 0.01 #when writing cq instance out/converting to STL file. Keep low 
EXPORT_ANGULAR_TOLERANCE = 0.05
 
ROUND_DECIMALS = 2  # for merging tessellation seam vertices after STL export

## Load hailstone mesh, repair it

In [6]:
print ('loading hailstone mesh, repairing')

mesh = trimesh.load(INPUT_STL, force="mesh")

#needs to be watertight but also check face winding and volume to fix.
if not mesh.is_watertight:
    v, f = mf.clean_from_arrays(mesh.vertices, mesh.faces)
    mesh = trimesh.Trimesh(v, f)

if not mesh.is_winding_consistent:
    trimesh.repair.fix_winding(mesh)

if mesh.volume < 0:
    mesh.invert()    
    
mesh.process(validate=True)
print ("after fixing, we can say that: is watertight:", mesh.is_watertight, "is_volume:", mesh.is_volume,
      "volume:", mesh.volume)

assert mesh.is_volume, "hailstone mesh is not a valid volume - curse the computer here"


loading hailstone mesh, repairing
after fixing, we can say that: is watertight: True is_volume: True volume: 45197.18969452773


## Generate the screw (threaded collar) and receiver

In [18]:
print ('making the screw files')

plug_thread = PlasticBottleThread(size=THREAD_SIZE, external=True,
    manufacturingCompensation=MANUFACTURING_COMP)

socket_thread = PlasticBottleThread(size=THREAD_SIZE, external=False,
    manufacturingCompensation=MANUFACTURING_COMP)

# Sanity check on bottle thread design - for an external (plug) thread the root_radius
# value should be the smaller of the two. (It's the valley between the screw ridges.)
# For an internal thread (socket) root_radius should be the larger of the two (where it
# goes into the bore wall). If either assertion fails, the root/apex convention is flipped 
# from what's expected, so will need to switch it.
print("plug   root/apex/pitch:", plug_thread.root_radius, plug_thread.apex_radius, 
      plug_thread.pitch)
print("socket root/apex/pitch:", socket_thread.root_radius, socket_thread.apex_radius, 
      socket_thread.pitch)
assert plug_thread.root_radius < plug_thread.apex_radius
assert socket_thread.root_radius > socket_thread.apex_radius

# also, the "pitch" of a screw is how deep the screw had has to go in order to make one
# full turn. It needs to be able to make at least one full turn (ideally more), so check
# that thread length (screw length) is larger than the screw pitch.
assert THREAD_LENGTH > plug_thread.pitch, "THREAD_LENGTH should exceed one full pitch"

#check that the generated cq instances are valid ones so far
print("plug_thread.isValid():", plug_thread.isValid())
print("socket_thread.isValid():", socket_thread.isValid())


OVERLAP = 0.2  # mm - core intentionally overlaps INTO the thread ridge's
                # material span, so the fuse has genuine 3D overlap to merge
                # rather than two shells that only touch along a coincident
                # surface (which pymeshfix's remove_smallest_components would
                # then delete outright as a disconnected "small" component)


# Add the "core" to the screw (and the wall around the screw hole/socket) so it isn't just
# a floating spiral ridge. The screw core extends to the valleys of the screw (root_radius).
plug_core = cq.Workplane("XY").circle(plug_thread.root_radius + OVERLAP).extrude(THREAD_LENGTH)
# Socket core is a hole cut through a cylinder, where the root radius matches the valleys 
# in the socket wall.
socket_core = (cq.Workplane("XY").circle(socket_thread.root_radius + WALL_THICKNESS)
               .circle(socket_thread.root_radius - OVERLAP).extrude(THREAD_LENGTH))


#check again that the generated cq instances are valid ones so far
print("plug_core.isValid():", plug_thread.isValid())
print("socket_core.isValid():", socket_thread.isValid())


#Fuse the core/hole wall to the screw threads themselves
plug_collar_cq = plug_thread.fuse(plug_core.val())
socket_collar_cq = socket_thread.fuse(socket_core.val())
# and once again check for validity
print("plug_collar_cq.isValid():", plug_collar_cq.isValid())
print("socket_collar_cq.isValid():", socket_collar_cq.isValid())


#Some cleaning to help with validity
plug_collar_cq = plug_collar_cq.clean()
socket_collar_cq = socket_collar_cq.clean()
print("after .clean() - plug valid:", plug_collar_cq.isValid(),
      "socket valid:", socket_collar_cq.isValid())


#finally, export
cq.exporters.export(plug_collar_cq, "plug_thread_collar.stl",
                     tolerance=EXPORT_TOLERANCE, angularTolerance=EXPORT_ANGULAR_TOLERANCE)
cq.exporters.export(socket_collar_cq, "socket_thread_collar.stl",
                     tolerance=EXPORT_TOLERANCE, angularTolerance=EXPORT_ANGULAR_TOLERANCE)
print("exported plug_thread_collar.stl and socket_thread_collar.stl")



making the screw files
plug   root/apex/pitch: 10.465 11.535 3.175
socket root/apex/pitch: 12.139999999999999 11.069999999999999 3.175
plug_thread.isValid(): True
socket_thread.isValid(): True
plug_core.isValid(): True
socket_core.isValid(): True
plug_collar_cq.isValid(): True
socket_collar_cq.isValid(): True
after .clean() - plug valid: True socket valid: True
exported plug_thread_collar.stl and socket_thread_collar.stl


## Finish conversion of CQ screw threaded objects to trimesh meshes

In [ ]:
print ('finish converting cq instances to trimesh')

def cq_to_trimesh(cq_solid, tol, ang_tol):
    verts, faces = cq_solid.tessellate(tol, ang_tol)
    v = np.array([(p.x, p.y, p.z) for p in verts], dtype=np.float64)
    f = np.array(faces, dtype=np.int64)
    return trimesh.Trimesh(vertices=v, faces=f, process=True)
 
plug_collar = cq_to_trimesh(plug_collar_cq, EXPORT_TOLERANCE, EXPORT_ANGULAR_TOLERANCE)
socket_collar = cq_to_trimesh(socket_collar_cq, EXPORT_TOLERANCE, EXPORT_ANGULAR_TOLERANCE)

print (socket_collar.bounds)
print (socket_thread.root_radius + WALL_THICKNESS)
print (plug_collar.bounds)
print (THREAD_LENGTH)

#check if it's a valid volume now
for name, collar in [("plug_collar", plug_collar), ("socket_collar", socket_collar)]:
    print(f"{name} raw - watertight: {collar.is_watertight}, is_volume: {collar.is_volume}")
    



#plot them just to see
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(socket_collar)

p.subplot(0, 1)
p.add_mesh(plug_collar)

p.show()

finish converting cq instances to trimesh
[[-14.14 -14.14   0.  ]
 [ 14.14  14.14   7.  ]]
14.139999999999999
[[-11.53427753 -11.53500011  -0.08258013]
 [ 11.53500011  11.53481954   7.        ]]
7.0
plug_collar raw - watertight: True, is_volume: True
socket_collar raw - watertight: True, is_volume: True


Widget(value='<iframe src="http://localhost:55611/index.html?ui=P_0x314b88710_0&reconnect=auto" class="pyvista…

## Do the hailstone slicing

In [9]:
print ('slicing hailstone in half')
top = trimesh.intersections.slice_mesh_plane(mesh, plane_normal=CUT_PLANE_NORMAL, 
                                             plane_origin=CUT_PLANE_POINT, cap=True)

bottom = trimesh.intersections.slice_mesh_plane(mesh, plane_normal=-CUT_PLANE_NORMAL, 
                                                plane_origin=CUT_PLANE_POINT, cap=True)

#make sure these meshes are both valid
print("top   - watertight:", top.is_watertight, "is_volume:", top.is_volume, "volume:", top.volume)
print("bottom- watertight:", bottom.is_watertight, "is_volume:", bottom.is_volume, "volume:", bottom.volume)
assert top.is_volume and bottom.is_volume, "slicing did not produce valid capped volumes, please howl at the moon"

# remove a cylindrical hole the size of socket_thread.root_radius + WALL_THICKNESS - OVERLAP from top
# The socket_thread.root_radius + WALL_THICKNESS is how big socket_collar is, the -OVERLAP is to ensure the objects overlap
cavity = trimesh.creation.cylinder(subdivisions=4, radius=socket_thread.root_radius + WALL_THICKNESS - OVERLAP,
                                   height = THREAD_LENGTH-OVERLAP)
cavity.apply_translation(np.array([0.0, 0.0, (THREAD_LENGTH-OVERLAP)/2.]))
 
top = trimesh.boolean.difference([top, cavity], engine="manifold")
print("top - cavity is watertight:", top.is_watertight, "is_volume:", 
      top.is_volume, "volume:", top.volume)


#make a plot to be certain it worked
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top, opacity=0.8, color='light_blue')
p.add_mesh(socket_collar, color='red')

p.subplot(0, 1)
p.add_mesh(bottom, opacity=0.3, color='light_blue')
p.add_mesh(plug_collar, color='red')

p.show()

slicing hailstone in half
top   - watertight: True is_volume: True volume: 22598.594847263867
bottom- watertight: True is_volume: True volume: 22598.594847263867
top - cavity is watertight: True is_volume: True volume: 18473.91566803151


Widget(value='<iframe src="http://localhost:55611/index.html?ui=P_0x33b751810_1&reconnect=auto" class="pyvista…

## Attach the screw and threads

In [10]:
print ('attaching screw and socket')
plug_collar.apply_translation(CAVITY_CENTER)
socket_collar.apply_translation(CAVITY_CENTER)
 
bottom = trimesh.boolean.union([bottom, plug_collar], engine="manifold")
print("bottom + plug, watertight:", bottom.is_watertight, "is_volume:", bottom.is_volume)
 
top = trimesh.boolean.union([top, socket_collar], engine="manifold")
print("top + socket collar, watertight:", top.is_watertight, "is_volume:", top.is_volume)
 
assert top.is_volume and bottom.is_volume, "collar attach step broke watertightness - stop here and reconsider your life choices"


#make a plot to be certain it worked
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top)

p.subplot(0, 1)
p.add_mesh(bottom)

p.show()


attaching screw and socket
bottom + plug, watertight: True is_volume: True
top + socket collar, watertight: True is_volume: True


Widget(value='<iframe src="http://localhost:55611/index.html?ui=P_0x33b788a10_2&reconnect=auto" class="pyvista…

## Remove the spherical hole for instrumentation from each half

In [11]:
print ('making instrumentation hole')
cavity = trimesh.creation.icosphere(subdivisions=4, radius=CAVITY_RADIUS)
cavity.apply_translation(CAVITY_CENTER)
 
top = trimesh.boolean.difference([top, cavity], engine="manifold")
print("top - cavity is watertight:", top.is_watertight, "is_volume:", 
      top.is_volume, "volume:", top.volume)
 
bottom = trimesh.boolean.difference([bottom, cavity], engine="manifold")
print("bottom- cavity is watertight:", bottom.is_watertight, "is_volume:", 
      bottom.is_volume, "volume:", bottom.volume)


#make a plot to be certain it worked
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top)

p.subplot(0, 1)
p.add_mesh(bottom)

p.show()

print("socket root/apex:", socket_thread.root_radius, socket_thread.apex_radius)
print("CAVITY_RADIUS:", CAVITY_RADIUS)

making instrumentation hole
top - cavity is watertight: True is_volume: True volume: 19504.850578582715
bottom- cavity is watertight: True is_volume: True volume: 22229.55771807816


Widget(value='<iframe src="http://localhost:55611/index.html?ui=P_0x346698350_3&reconnect=auto" class="pyvista…

socket root/apex: 12.139999999999999 11.069999999999999
CAVITY_RADIUS: 9.0


## Finally, write the blasted things out

In [12]:
print ('writing out')
for name, half, path in [("top", top, OUTPUT_TOP_STL), ("bottom", bottom, OUTPUT_BOTTOM_STL)]:
    if not half.is_watertight:
        print ('fixing ', name)
        v, f = mf.clean_from_arrays(half.vertices, half.faces)
        half = trimesh.Trimesh(v, f)
    half.export(path)
    print(f"{path}: watertight={half.is_watertight}, volume={half.volume:.1f} mm^3")

writing out
ellipsoid_surf_top.stl: watertight=True, volume=19504.9 mm^3
ellipsoid_surf_bottom.stl: watertight=True, volume=22229.6 mm^3


## Debugging plots

In [13]:
top_check = pv.wrap(top)  # or pv.read(OUTPUT_TOP_STL) to check the actual exported file
clipped = top_check.clip(normal='x', origin=(0, 0, 0))

plotter = pv.Plotter()
plotter.add_mesh(clipped, color='lightblue', show_edges=True)
plotter.add_mesh(clipped.extract_feature_edges(), color='red', line_width=2)
plotter.show()

Widget(value='<iframe src="http://localhost:55611/index.html?ui=P_0x346682910_4&reconnect=auto" class="pyvista…

In [19]:
bottom_check = pv.wrap(bottom)
clipped_b = bottom_check.clip(normal='x', origin=(0, 0, 0))
plotter = pv.Plotter()
plotter.add_mesh(clipped_b, color='lightgreen', show_edges=True)
plotter.show()

overlap = trimesh.boolean.intersection([plug_collar, socket_collar], engine="manifold")
print("interference volume:", overlap.volume if overlap is not None else 0)

plug_collar.export('plug_collar_test.stl')
socket_collar.export('socket_collar_test.stl')

Widget(value='<iframe src="http://localhost:55611/index.html?ui=P_0x342dea090_9&reconnect=auto" class="pyvista…

interference volume: 11.612237737848735


b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xfa\x10\x00\x00z\xfb\x7f\xbf\x90\x7f@\xbc\x00\x00\x00\x004\x121A,+\x85>\xefq\x1e@\xb9\x1e1A\x03\xf9\xe63\n\xd7\xcb?\xb9\x1e1A\x03\xf9\xe63e\xaf\x1d@\x00\x00z\xfb\x7f\xbf\x90\x7f@\xbc\x00\x00\x00\x804\x121A,+\x85>\xefq\x1e@4\x121A,+\x85>\x1f\\\xcd?\xb9\x1e1A\x03\xf9\xe63\n\xd7\xcb?\x00\x00L\xd7\x7f\xbf\xc0V\x10\xbd\x00\x00\x00\x80\xa9\xec0A\xc3!\x05?y4\x1f@\xa9\xec0A\xc3!\x05?3\xe1\xce?4\x121A,+\x85>\x1f\\\xcd?\x00\x00L\xd7\x7f\xbf\xc0V\x10\xbd\x00\x00\x00\x00\xa9\xec0A\xc3!\x05?y4\x1f@4\x121A,+\x85>\x1f\\\xcd?4\x121A,+\x85>\xefq\x1e@\x00\x00\xf5\x8e\x7f\xbf\xffyp\xbd\x00\x00\x00\x80\x1c\xae0A\x1d\x9bG?\x03\xf7\x1f@\x1c\xae0A\x1d\x9bG?Gf\xd0?\xa9\xec0A\xc3!\x05?3\xe1\xc